<a href="https://colab.research.google.com/github/handikanurichsan/Beginner-cpp/blob/main/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [241]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [242]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


Data source import complete.


In [243]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Load Data and Split

In [244]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


xtrain = pd.read_csv(os.path.join(titanic_path, 'train.csv'))
xtest = pd.read_csv(os.path.join(titanic_path, 'test.csv'))

y=xtrain.Survived
xtrain.drop(['Survived'], axis=1, inplace=True)

x_trainfull, x_val, y_trainfull, y_val = train_test_split(xtrain, y, test_size=0.2, random_state=0)

# Differs the Num and Cat Columns

In [245]:
num_cols = x_train.select_dtypes(exclude=['object']).columns
cat_cols = x_train.select_dtypes(include=['object']).columns

fullcols = num_cols.append(cat_cols)

x_train = x_trainfull[fullcols].copy()
x_val = x_val[fullcols].copy()
x_test = xtest[fullcols].copy()

In [246]:
print(num_cols)
print(cat_cols)
print(fullcols)
print(f'\nx_train shape: {xtrain.shape}\n')
print(f'x_train columns: {xtrain.columns}')

Index(['PassengerId', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')
Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='object')
Index(['PassengerId', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Name', 'Sex',
       'Ticket', 'Cabin', 'Embarked'],
      dtype='object')

x_train shape: (891, 11)

x_train columns: Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


In [247]:
x_train.head()

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Name,Sex,Ticket,Cabin,Embarked
140,141,3,NaN,0,2,15.2458,"Boulos, Mrs. Joseph (Sultana)",female,2678,NaN,C
439,440,2,31.0,0,0,10.5000,"Kvillner, Mr. Johan Henrik Johannesson",male,C.A. 18723,NaN,S
817,818,2,31.0,1,1,37.0042,"Mallet, Mr. Albert",male,S.C./PARIS 2079,NaN,C
378,379,3,20.0,0,0,4.0125,"Betros, Mr. Tannous",male,2648,NaN,C
491,492,3,21.0,0,0,7.2500,"Windelov, Mr. Einar",male,SOTON/OQ 3101317,NaN,S


In [248]:
x_train.isnull().sum()

,0
PassengerId,0
Pclass,0
Age,141
SibSp,0
Parch,0
Fare,0
Name,0
Sex,0
Ticket,0
Cabin,549


In [249]:
print(num_cols)
print(cat_cols)

Index(['PassengerId', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')
Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='object')


In [250]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

numericalTransformer = SimpleImputer(strategy='median')


categoricalTransformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numericalTransformer, num_cols),
        ('cat', categoricalTransformer, cat_cols)
    ])

myPipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', XGBClassifier(n_estimators=500, learning_rate=0.05,
                                  max_depth=4,random_state=42,eval_metric='logloss'))])


myPipeline.fit(x_train,y_train)

preds = myPipeline.predict(x_val)

print('Accuracy:', accuracy_score(y_val, preds))



Accuracy: 0.8603351955307262


# Create Pediction Test

In [251]:
prediction = myPipeline.predict(x_test)

In [252]:
# Save test predictions to file
output = pd.DataFrame({'Id': x_test.index,
                       'Survived': prediction})
output.to_csv('submission.csv', index=False)